# 03. 통합 게이트와 최종 판정

## 핵심 원칙

**파일을 합칠 수 있다**와 **같이 학습해도 된다**는 다른 결론입니다. 기술 게이트를 통과한 뒤에도 source-only 기준선, 교차-domain 성능, mixed 모델의 source별 성능을 확인해야 합니다.

이 노트북은 앞 단계 결과를 모아 PASS/FAIL/REVIEW/PENDING을 판정하고, 원본을 복사하지 않는 canonical manifest 초안을 만듭니다.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    """VS Code가 프로젝트 폴더/노트북 폴더 중 어디를 cwd로 잡아도 동작하게 합니다."""
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "project_config.json").is_file() and (candidate / "ctqa_utils.py").is_file():
            return candidate
    raise FileNotFoundError("project_config.json이 있는 CT_Data_Quality_Notebooks 폴더를 VS Code에서 여세요.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from ctqa_utils import *

CONFIG = json.loads((PROJECT_ROOT / "project_config.json").read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"])
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("프로젝트:", PROJECT_ROOT)
print("데이터:", DATA_ROOT)
print("결과 저장:", RESULTS)

프로젝트: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합
데이터: C:\Users\duddl\OneDrive\바탕 화면\CTdata
결과 저장: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results


In [2]:
required = [
    RESULTS / "00_pair_manifest.csv",
    RESULTS / "00_inventory_summary.csv",
    RESULTS / "01_geometry_checks.csv",
    RESULTS / "02_value_label_checks.csv",
    RESULTS / "02_physionet_slice_consistency.csv",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError("앞 노트북을 순서대로 실행하세요. 누락:\n" + "\n".join(missing))

pairs = pd.read_csv(required[0])
inventory = pd.read_csv(required[1])
geometry = pd.read_csv(required[2])
values = pd.read_csv(required[3])
consistency = pd.read_csv(required[4])
checksum_path = RESULTS / "00_physionet_sha256.csv"
checksum = pd.read_csv(checksum_path) if checksum_path.is_file() else pd.DataFrame()

## 1. 지표 계산

source별 중앙 z-spacing에서 네 배 이상 벗어나면 3D 모델 전 확인이 필요한 강한 이상치로 표시합니다. group ID 중복은 환자/검사 누수 방지를 위해 같은 split에 묶어야 합니다.

In [3]:
z_outlier = pd.Series(False, index=geometry.index)
for source, idx in geometry.groupby("source").groups.items():
    median_z = float(geometry.loc[idx, "z_spacing_mm"].median())
    ratio = geometry.loc[idx, "z_spacing_mm"] / median_z
    z_outlier.loc[idx] = (ratio < 0.25) | (ratio > 4.0)

duplicate_groups = pairs.groupby(["source", "group_id"]).size()
duplicate_groups = duplicate_groups[duplicate_groups > 1]

metrics = {
    "total_pairs": int(len(pairs)),
    "pair_missing": int(inventory.ct_without_mask.sum() + inventory.mask_without_ct.sum()),
    "checksum_mismatch": int((~checksum["match"].astype(bool)).sum()) if not checksum.empty else None,
    "shape_mismatch": int((~geometry.shape_match.astype(bool)).sum()),
    "spacing_mismatch": int((~geometry.spacing_match.astype(bool)).sum()),
    "affine_mismatch": int((~geometry.affine_match.astype(bool)).sum()),
    "transform_code_mismatch": int((~geometry.header_transform_code_match.astype(bool)).sum()),
    "z_spacing_outliers": int(z_outlier.sum()),
    "nonfinite_or_constant": int((values.ct_finite_fraction < 1).sum() + values.ct_constant.astype(bool).sum()),
    "invalid_label_cases": int((values.invalid_label_count > 0).sum()),
    "intensity_encoding_candidates": int(values.ct_encoding_candidate.astype(bool).sum()),
    "physionet_slice_mismatch": int(consistency.positive_slice_mismatch.fillna(0).sum()),
    "duplicate_split_groups": int(len(duplicate_groups)),
    "positive_cases": int((~values.empty_mask.astype(bool)).sum()),
    "negative_cases": int(values.empty_mask.astype(bool).sum()),
}
print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "total_pairs": 267,
  "pair_missing": 0,
  "checksum_mismatch": 0,
  "shape_mismatch": 0,
  "spacing_mismatch": 0,
  "affine_mismatch": 4,
  "transform_code_mismatch": 191,
  "z_spacing_outliers": 1,
  "nonfinite_or_constant": 0,
  "invalid_label_cases": 0,
  "intensity_encoding_candidates": 1,
  "physionet_slice_mismatch": 0,
  "duplicate_split_groups": 1,
  "positive_cases": 228,
  "negative_cases": 39
}


## 2. 통합 게이트

자동으로 알 수 없는 라이선스 승인, 변환 회귀 테스트, 모델 성능은 PENDING/REVIEW로 남겨둡니다. 이를 PASS로 임의 변경하면 안 됩니다.

In [4]:
def status_zero(value):
    return "PASS" if value == 0 else "FAIL"

gates = [
    {"gate": "G1 권한·라이선스", "test": "두 source 허용 목적·변경·공유 범위 확인", "pass_criteria": "기관/사용자 승인 기록 존재", "result": "BHSD/PhysioNet 조건 별도 검토 필요", "status": "REVIEW"},
    {"gate": "G2 파일 인벤토리", "test": "CT-mask 1:1 대응", "pass_criteria": "누락 0", "result": metrics["pair_missing"], "status": status_zero(metrics["pair_missing"])},
    {"gate": "G3 무결성", "test": "제공 SHA256 검증", "pass_criteria": "불일치 0", "result": metrics["checksum_mismatch"], "status": "PASS" if metrics["checksum_mismatch"] == 0 else "REVIEW"},
    {"gate": "G4 값 복원", "test": "finite·상수·인코딩 후보", "pass_criteria": "문제 0 또는 승인 교정 100%", "result": metrics["nonfinite_or_constant"] + metrics["intensity_encoding_candidates"], "status": status_zero(metrics["nonfinite_or_constant"] + metrics["intensity_encoding_candidates"])},
    {"gate": "G5 공간 정합", "test": "shape·spacing·qform/sform·effective affine", "pass_criteria": "material affine 불일치 0 또는 승인 보정 100%; code 차이는 문서화", "result": f"shape={metrics['shape_mismatch']}, spacing={metrics['spacing_mismatch']}, code={metrics['transform_code_mismatch']}, material_affine={metrics['affine_mismatch']}", "status": status_zero(metrics["shape_mismatch"] + metrics["spacing_mismatch"] + metrics["affine_mismatch"])},
    {"gate": "G6 이상치", "test": "z-spacing 등 robust outlier", "pass_criteria": "원인·처리 규칙 확정", "result": metrics["z_spacing_outliers"], "status": "PASS" if metrics["z_spacing_outliers"] == 0 else "REVIEW"},
    {"gate": "G7 라벨 일관성", "test": "허용 label과 PhysioNet CSV 비교", "pass_criteria": "불일치 0", "result": metrics["invalid_label_cases"] + metrics["physionet_slice_mismatch"], "status": status_zero(metrics["invalid_label_cases"] + metrics["physionet_slice_mismatch"])},
    {"gate": "G8 ontology", "test": "공통 target mapping", "pass_criteria": "모든 source label의 명시적 mapping", "result": "binary: mask>0 가능 / 5종 voxel target 불가", "status": "PASS_BINARY_ONLY"},
    {"gate": "G9 split 누수", "test": "환자·검사·근접 중복의 split 교차", "pass_criteria": "group overlap 0, 공식 test 보존", "result": metrics["duplicate_split_groups"], "status": "REVIEW" if metrics["duplicate_split_groups"] else "PASS"},
    {"gate": "G10 변환 회귀", "test": "resampling/windowing 후 label·부피·overlay", "pass_criteria": "예상 외 label 0, 병변 소실 0", "result": "전처리 구현 후 실행", "status": "PENDING"},
    {"gate": "G11 domain shift", "test": "전처리 후 source classifier", "pass_criteria": "AUC<0.75 권장", "result": "모델 실험 전", "status": "PENDING"},
    {"gate": "G12 모델 비열등성", "test": "source-only vs mixed locked test", "pass_criteria": "source별 Dice 저하 ≤2%p, 최악 subgroup ≤5%p", "result": "모델 실험 전", "status": "PENDING"},
]
gates_df = pd.DataFrame(gates)
gates_df.to_csv(RESULTS / "03_integration_gates.csv", index=False, encoding="utf-8-sig")
print(gates_df[["gate", "result", "status"]].to_string(index=False))

            gate                                          result           status
      G1 권한·라이선스                      BHSD/PhysioNet 조건 별도 검토 필요           REVIEW
      G2 파일 인벤토리                                               0             PASS
          G3 무결성                                               0             PASS
         G4 값 복원                                               1             FAIL
        G5 공간 정합 shape=0, spacing=0, code=191, material_affine=4             FAIL
          G6 이상치                                               1           REVIEW
       G7 라벨 일관성                                               0             PASS
     G8 ontology          binary: mask>0 가능 / 5종 voxel target 불가 PASS_BINARY_ONLY
     G9 split 누수                                               1           REVIEW
       G10 변환 회귀                                     전처리 구현 후 실행          PENDING
G11 domain shift                                         모델 실험 전          PENDING
     G12 모델 비열등성

## 3. Canonical manifest 초안

두 데이터셋을 물리적으로 복사·병합하지 않습니다. 경로와 변환 규칙만 가진 manifest를 만들고 dataloader에서 `mask > 0` 이진 변환을 적용하는 방식이 안전합니다.

In [5]:
canonical = pairs.copy()
canonical["task"] = "binary_hemorrhage_segmentation"
canonical["binary_label_rule"] = "scaled_mask > 0"
canonical["multiclass_available"] = canonical.source.eq("BHSD")
canonical["multiclass_rule"] = np.where(canonical.source.eq("BHSD"), "labels 0..5", "missing")
canonical["split_status"] = "UNASSIGNED_GROUPED_SPLIT_REQUIRED"
canonical["technical_status"] = np.where(canonical.source.eq("BHSD"), "REQUIRES_SPATIAL_AND_INTENSITY_REVIEW", "BINARY_READY_AFTER_LOADER_UNIT_TEST")
canonical.to_csv(RESULTS / "03_canonical_manifest_draft.csv", index=False, encoding="utf-8-sig")
print(canonical.groupby(["source", "technical_status"]).size().to_string())

source            technical_status                     
BHSD              REQUIRES_SPATIAL_AND_INTENSITY_REVIEW    192
PhysioNet_CT_ICH  BINARY_READY_AFTER_LOADER_UNIT_TEST       75


## 4. 최종 결론

현재 자동 검사 결과만으로는 바로 통합 학습을 시작하지 않습니다. FAIL/REVIEW 항목을 처리하고 G10–G12 실험까지 통과해야 최종 GO입니다.

In [6]:
decision = {
    "binary_integration_design": "CONDITIONALLY_COMPATIBLE",
    "binary_training_now": "HOLD",
    "five_class_voxel_integration": "NOT_COMPATIBLE",
    "reason": [
        "BHSD 공간 헤더/강도 후보/spacing 검토가 남아 있음",
        "PhysioNet은 binary mask이므로 BHSD 5종 voxel label과 직접 합칠 수 없음",
        "변환 회귀·domain shift·mixed model 비열등성 실험이 아직 PENDING",
    ],
    "go_when": [
        "G1~G10 기술·법적 게이트가 PASS",
        "source classifier AUC<0.75 권장 또는 domain 대응 근거 확보",
        "mixed 모델의 source별 Dice 저하가 각 source-only 대비 2%p 이하",
        "최악 subgroup 저하가 5%p 이하",
    ],
}
write_json(RESULTS / "03_final_decision.json", decision)
print(json.dumps(decision, ensure_ascii=False, indent=2))
print("\n모든 노트북 실행 완료 — results 폴더의 CSV/JSON/PNG를 검토하세요.")

{
  "binary_integration_design": "CONDITIONALLY_COMPATIBLE",
  "binary_training_now": "HOLD",
  "five_class_voxel_integration": "NOT_COMPATIBLE",
  "reason": [
    "BHSD 공간 헤더/강도 후보/spacing 검토가 남아 있음",
    "PhysioNet은 binary mask이므로 BHSD 5종 voxel label과 직접 합칠 수 없음",
    "변환 회귀·domain shift·mixed model 비열등성 실험이 아직 PENDING"
  ],
  "go_when": [
    "G1~G10 기술·법적 게이트가 PASS",
    "source classifier AUC<0.75 권장 또는 domain 대응 근거 확보",
    "mixed 모델의 source별 Dice 저하가 각 source-only 대비 2%p 이하",
    "최악 subgroup 저하가 5%p 이하"
  ]
}

모든 노트북 실행 완료 — results 폴더의 CSV/JSON/PNG를 검토하세요.
